# Module Investigation
for phase 1 oysters 

Interested to know if there are a subset of genes that drive the relationship we see between modules and factors. Here, I'm going to pull genes within a significantly correlated module and look into their individual GE.

## 0. load libraries

In [ ]:
library(tidyverse)

library(pheatmap)
library(RColorBrewer)

# for gene ontology
library(rtracklayer)
library(clusterProfiler)
library(GO.db)

# for adding plots together 
library(cowplot)

## 1. read csvs

### module genes
generated in [WGCNA](https://github.com/jgmcdonough/CE24_RNA-seq/blob/main/analysis/diff_expression/phase1_v_phase1/wgcna/newRef_wgcna_p1.ipynb) for phase 1 oysters

In [2]:
geneInfo <- read.csv('/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase1_v_phase1/wgcna/outputs/p1.wgcna_GeneInfo.csv')
head(geneInfo)

,Gene,GO.terms,ModuleColor,GS.Actual_shell_growth_mg,GS.Actual_tissue_growth_mg,GS.Ratio_tissue_shell_mg,GS.P1_trtmt_code,GS.P1_temp_code,GS.P1_DO_code,GS.both,⋯,MMblue,MMgreenyellow,MMyellow,MMblack,MMbrown,MMpink,MMmagenta,MMtan,MMgreen,MMred
,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,LOC111099029,GO:0005261;GO:0005886;GO:0030001;GO:0098655,blue,0.39966747,0.282186775,0.06994163,0.1492213,0.385003047,-0.02872849,0.12634487,⋯,0.3336787,0.09526861,-0.03750345,-0.01178295,-0.04856269,0.115416013,-0.46324802,-0.19648759,0.05033161,-0.04085780
2,LOC111099037,GO:0000978;GO:0000981;GO:0006355,black,-0.02461599,0.002751307,-0.03894892,-0.4217885,-0.363072643,-0.28138410,-0.24659525,⋯,-0.2452065,-0.14257739,-0.16774119,0.34293007,0.31439017,-0.065239565,0.07719106,-0.15380042,0.22565913,0.02356195
3,LOC111099039,GO:0004930;GO:0005886;GO:0007189,blue,0.44340350,0.303838275,-0.01706141,-0.2141764,0.221972568,-0.34604876,-0.04315846,⋯,0.1248501,0.26055230,0.01693705,0.20241978,0.34095291,0.002859469,-0.37935688,-0.49934818,0.05801009,-0.35374473
4,LOC111099041,GO:0016020;GO:0022857;GO:0055085,red,-0.41098524,-0.310348052,-0.04403876,0.3171945,0.004370064,0.34594159,0.31502257,⋯,-0.4530072,-0.51091263,-0.31182848,-0.07876548,0.07063991,0.365824475,0.50638260,0.32368892,0.40288267,0.68022795
5,LOC111099050,GO:0005515;GO:0006886;GO:0031462;GO:0043161;GO:1990756,green,-0.35176148,-0.282846525,-0.03894228,-0.0592647,-0.254057873,0.06198487,-0.01882029,⋯,-0.3888532,-0.68292735,-0.49529157,0.14867298,0.47087567,-0.253669949,0.40577513,0.03737518,0.72387889,0.49103455
6,LOC111099053,GO:0001731;GO:0003729;GO:0003743;GO:0005850;GO:0006413;GO:0031369,blue,0.43025668,0.181368789,-0.15299170,-0.3250255,0.370840965,-0.54214175,-0.10090793,⋯,0.7387194,0.17197754,0.25907960,0.07602426,-0.20551234,-0.602026100,-0.33485347,-0.05929480,-0.20905792,-0.29068175


### vst
generated in [DESeq analysis](https://github.com/jgmcdonough/CE24_RNA-seq/blob/main/analysis/diff_expression/phase1_v_phase1/filter_deseq_p1.v.p1.ipynb) for phase 1 oysters

In [3]:
vst <- read.csv('/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase1_v_phase1/deseq_res/prefilter_deseq/p1_vst.csv')
head(vst)

,X,B1_Nu_O03,B2_Nu_O12,B4_Nu_O32,B5_Nu_O36,B6_Nu_O47,C1_Nu_W01,C1_Nu_W05,C2_Nu_W15,C3_Nu_W21,⋯,H3_Nu_B18,H4_Nu_B28,H5_Nu_B35,H6_Nu_B45,W1_Nu_G02,W2_Nu_G15,W3_Nu_G21,W4_Nu_G27,W6_Nu_G41,W6_Nu_G45
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,LOC144621260,11.217975,11.934767,11.224763,11.385998,11.491494,11.330413,11.619094,11.599610,10.935631,⋯,11.695721,11.157977,11.658281,11.998989,11.369555,11.484610,11.158177,11.698655,11.463655,11.597431
2,LOC144621269,9.238570,14.233747,13.248661,6.734822,7.542979,13.067731,11.252050,11.655097,11.695209,⋯,13.172456,11.238048,8.354464,15.534626,13.113269,11.584691,11.875265,13.441989,6.077703,12.341027
3,LOC111120925,6.162650,6.596958,6.709412,6.097391,6.365111,6.642652,8.868688,8.893482,8.281309,⋯,8.933040,6.529973,8.449891,8.330386,6.304683,8.790027,6.259740,6.294474,5.806396,6.516750
4,LOC144621283,9.891478,10.157309,10.265604,9.957131,10.370479,10.049745,9.730975,10.158138,10.091135,⋯,10.091122,9.802983,9.406153,9.831201,9.297184,10.572628,10.579026,9.655386,10.115723,10.146704
5,LOC144621276,13.540965,13.766144,14.145528,13.224735,13.084951,13.044656,13.810034,13.596797,13.309595,⋯,13.929735,13.686105,12.877924,13.996777,12.961637,13.892552,14.181140,13.333327,13.928741,13.550413
6,LOC111115920,8.717293,8.463000,8.350118,8.954966,9.308119,8.469442,7.913170,8.149178,8.357417,⋯,7.567075,8.159515,8.690883,8.016850,7.795664,8.011123,8.435333,8.088760,8.136625,7.770861


### meta data

In [4]:
growth <- read.csv('/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/metaData/growth_phase1_weights.csv')

# add lead
growth$Tag_num <- sprintf("%02d", growth$Tag_num)

# need to convert sample names to be consistent with my naming convention
growth$Sample <- paste0(substr(growth$Phase_1_treat, 1, 1), growth$Phase_1_rep, '_Nu_', growth$Tag_color, growth$Tag_num)

growth <- growth %>% dplyr::select(Sample, 
                            Phase_1_treat, 
                            Phase_1_temp, 
                            Phase_1_DO, 
                            Actual_tissue_growth_mg, 
                            Actual_shell_growth_mg, 
                            Ratio_tissue_shell_mg)

head(growth)

,Sample,Phase_1_treat,Phase_1_temp,Phase_1_DO,Actual_tissue_growth_mg,Actual_shell_growth_mg,Ratio_tissue_shell_mg
,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>
1,B1_Nu_O01,Both,Warm,Hyp,21.92322,42.27678,0.518564091
2,B1_Nu_O02,Both,Warm,Hyp,42.34732,63.65268,0.665287306
3,B1_Nu_O03,Both,Warm,Hyp,27.91340,77.58660,0.359770888
4,B1_Nu_O04,Both,Warm,Hyp,148.21004,157.38996,0.941674043
5,B1_Nu_O05,Both,Warm,Hyp,-40.02418,-19.47582,2.055070339
6,B1_Nu_O06,Both,Warm,Hyp,117.50070,181.29930,0.64810344


In [5]:
# keep only growth for genetics oysters
growth <- growth[growth$Sample %in% colnames(vst), ]

rownames(growth) <- NULL

dim(growth)
head(growth)

[1] 23  7

,Sample,Phase_1_treat,Phase_1_temp,Phase_1_DO,Actual_tissue_growth_mg,Actual_shell_growth_mg,Ratio_tissue_shell_mg
,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>
1,B1_Nu_O03,Both,Warm,Hyp,27.91340,77.58660,0.359770888
2,B2_Nu_O12,Both,Warm,Hyp,80.37700,150.42300,0.534339828
3,B4_Nu_O32,Both,Warm,Hyp,105.36056,145.03944,0.726426964
4,B5_Nu_O36,Both,Warm,Hyp,25.22362,32.77638,0.769566987
5,B6_Nu_O47,Both,Warm,Hyp,-12.30496,117.80496,-0.104451969
6,C1_Nu_W01,Cont,Ambient,Norm,136.95688,232.44312,0.589205996


### gene annotations

In [8]:
annot <- as.data.frame(import('/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Cvirginica_genome/new_cvir_genome/GCF_053477285.1_ASM5347728v1_genomic.gtf')) %>%
dplyr::select(gene_id, description) %>%
na.omit()

head(annot)

,gene_id,description
,<chr>,<chr>
1,LOC144621260,protein O-mannosyl-transferase TMTC2-like
29,LOC144621269,uncharacterized LOC144621269
68,LOC111120925,mitochondrial amidoxime-reducing component 1-like
86,Trnae-cuc,transfer RNA glutamic acid (anticodon CUC)
89,Trnae-cuc_1,transfer RNA glutamic acid (anticodon CUC)
92,Trnae-cuc_2,transfer RNA glutamic acid (anticodon CUC)


#### for GO analysis

In [10]:
geneGO <- read.csv('/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Cvirginica_genome/annotations/newRef_geneGO.csv')
colnames(geneGO)[1] <- 'Gene'
colnames(geneGO)[2] <- 'GO.terms'

# One row for each gene and GO.term
term2gene <- geneGO %>%
  separate_rows(GO.terms, sep = ";") %>% # one row for each GO.term
  filter(!is.na(GO.terms) & GO.terms != "") %>% # remove genes without GO terms
  dplyr::select(term = GO.terms, gene = Gene) # rename columns for GO 

# 

head(term2gene)

term,gene
<chr>,<chr>
GO:0005261,LOC111099029
GO:0005886,LOC111099029
GO:0030001,LOC111099029
GO:0098655,LOC111099029
GO:0004930,LOC111099032
GO:0005886,LOC111099032


this becomes input for `enricher` down below

get term names for GO IDs

In [11]:
# Extract GO term descriptions
go_terms <- unique(term2gene$term)

# Get descriptions from GO.db
term2name <- data.frame(
  term = go_terms,
  name = sapply(go_terms, function(x) {
    tryCatch({
      Term(GOTERM[[x]])
    }, error = function(e) {
      NA_character_
    })
  })
)

# Remove NAs
term2name <- term2name[!is.na(term2name$name), ]

# View
head(term2name)  

,term,name
,<chr>,<chr>
GO:0005261,GO:0005261,monoatomic cation channel activity
GO:0005886,GO:0005886,plasma membrane
GO:0030001,GO:0030001,metal ion transport
GO:0098655,GO:0098655,monoatomic cation transmembrane transport
GO:0004930,GO:0004930,G protein-coupled receptor activity
GO:0007186,GO:0007186,G protein-coupled receptor signaling pathway


## 2. pull vst for sig. modules
for phase 1 oysters, only 3 modules were significantly correlated with factors:
1. red - negatively correlated with tissue growth
2. blue - positively correlated with shell growth and warm treatment
3. black - negatively correlated with temperature 

### red module
negatively correlated with tissue growth

plotting tissue growth

In [ ]:
growth$Phase_1_treat <- factor(growth$Phase_1_treat, levels = c('Cont', 'Hyp', 'Warm', 'Both'))

tissue_growth <- ggplot(growth, aes(x = Phase_1_treat, y = Actual_tissue_growth_mg, fill = Phase_1_treat)) +
geom_violin() +
geom_point() +
stat_summary(
    fun = "mean", 
    geom = "point", 
    shape = 8,      # asterick shape
    size = 3,        # Size of the point
    color = "yellow"  # Outline color
  ) +
scale_fill_brewer(palette = 'Set2') +
guides(fill = 'none') +
labs(x = 'Treatment',
     y = 'Tissue Growth (mg)',
    title = 'Tissue Growth') +
theme_bw(base_size = 15) +
theme(plot.title = element_text(hjust = 0.5))

tissue_growth

the black points represent the individual sample values, the yellow asterick represents the mean tissue growth for that treatment 

In [ ]:
# pull red genes
red_genes <- geneInfo %>% 
filter(ModuleColor == 'red') %>% 
pull(Gene)

length(red_genes) # 714 genes in the red module

In [ ]:
# vst for red genes
red_vst <- vst[vst$X %in% red_genes,]

dim(red_vst)
head(red_vst)

In [ ]:
# convert vst to long format
red_vst.long <- red_vst %>%
pivot_longer(cols = -X,
             names_to = 'Sample',
             values_to = 'vst') %>%
rename(Gene = X)

head(red_vst.long)

In [ ]:
# add growth info
red_vst_growth <- merge(red_vst.long, growth, by = 'Sample')
head(red_vst_growth)

#### plot growth vs. module expression

In [ ]:
ggplot(red_vst_growth, aes(x = Actual_tissue_growth_mg, y = vst)) +
geom_point() +
geom_smooth() +
theme_bw()

#### hub gene: LOC111135053

In [ ]:
red_hubGene <- red_vst_growth %>% filter(Gene == 'LOC111135053')

ggplot(red_hubGene, aes(x = vst, y = Actual_tissue_growth_mg)) +
geom_point() +
labs(x = 'VST Expression',
     y = 'Tissue Growth (mg)',
     title = 'Red Hub Gene LOC111135053') +
theme_bw(base_size = 15)

#### vst vs. growth 

In [ ]:
red_gs <- geneInfo %>% 
filter(ModuleColor == 'red') %>%
dplyr::select(Gene, GS.Actual_tissue_growth_mg, p.GS.GS.Actual_tissue_growth_mg)

# split by 

dim(red_gs)
head(red_gs)

now want to split this group into genes that are significant (pval < 0.05) and pos/neg correlated

In [ ]:
red_gs2 <- red_gs %>%
mutate(geneClass = case_when(
    p.GS.GS.Actual_tissue_growth_mg < 0.05 & GS.Actual_tissue_growth_mg > 0 ~ 'Sig Pos',
    p.GS.GS.Actual_tissue_growth_mg < 0.05 & GS.Actual_tissue_growth_mg < 0 ~ 'Sig Neg',
    TRUE ~ 'Not Sig'))

head(red_gs2)

In [ ]:
red_geneClass <- merge(red_vst_growth, red_gs2, by = 'Gene')
head(red_geneClass)

In [ ]:
options(repr.plot.height = 7.5, repr.plot.width = 10)

ggplot(red_geneClass, aes(x = Actual_tissue_growth_mg, y = vst)) +
geom_point() +
geom_smooth() +
facet_wrap(~geneClass, nrow = 1) +
theme_bw()

hmm so there are no genes that have a positive correlation *and* have a significant pvalue - which makes sense since this module has a negative correlation with tissue growth

pulling out the top 20 correlated genes and plotting their expression

In [ ]:
# grab only the significant negatively correlated genes
signeg_red <- red_gs2 %>% filter(geneClass == 'Sig Neg')

# order by correlation, take the top 20 genes
sig_red_top20 <- signeg_red[order(signeg_red$GS.Actual_tissue_growth_mg, decreasing = TRUE),][1:20,]

# pull vst and growth for samples for these 20 genes
red_gc_top20 <- red_geneClass[red_geneClass$Gene %in% sig_red_top20$Gene,]

# add gene annotation
colnames(red_gc_top20)[1] <- 'gene_id'
red_gc_top20 <- merge(red_gc_top20, annot, by = 'gene_id')

# check things look okay
length(unique(red_gc_top20$gene_id))
head(red_gc_top20)

In [ ]:
options(repr.plot.width = 17.5, repr.plot.height = 10)

ggplot(red_gc_top20, aes(x = Actual_tissue_growth_mg, y = vst, col = gene_id)) +
geom_point() +
geom_smooth() +
facet_wrap(~description) +
theme_bw()

seems like most of them have week trends (in terms of vst vs. growth)

In [ ]:
annot[annot$gene_id %in% red_gc_top20$Gene,]

most of the top 20 negatively correlated genes with tissue growth are uncharacterized. 

we do see a glycoprotein show up, and other transport/transferase related genes

### black module
negatively correlated with phase 1 temperature

In [ ]:
# pull black genes
black_genes <- geneInfo %>% 
filter(ModuleColor == 'black') %>% 
pull(Gene)

length(black_genes) # 528 genes in the red module

In [ ]:
# vst for red genes
black_vst <- vst[vst$X %in% black_genes,]

dim(black_vst)
head(black_vst)

In [ ]:
# convert vst to long format
black_vst.long <- black_vst %>%
pivot_longer(cols = -X,
             names_to = 'Sample',
             values_to = 'vst')

colnames(black_vst.long)[1] <- 'Gene'

head(black_vst.long)

In [ ]:
# add growth info
black_vst_growth <- merge(black_vst.long, growth, by = 'Sample')
head(black_vst_growth)

#### plot phase 1 temp vs. module expression

In [ ]:
options(repr.plot.height = 7.5, repr.plot.width = 7.5)

ggplot(black_vst_growth, aes(x = Phase_1_temp, y = vst, fill = Phase_1_temp)) +
geom_boxplot() +
scale_fill_brewer(palette = 'Set1', direction = -1) +
guides(fill = 'none') +
theme_bw(base_size = 15)

In [ ]:
options(repr.plot.height = 7.5, repr.plot.width = 7.5)

black_vst_growth$Phase_1_treat <- factor(black_vst_growth$Phase_1_treat, levels = c('Cont', 'Hyp', 'Warm', 'Both'))

ggplot(black_vst_growth, aes(x = Phase_1_treat, y = vst, fill = Phase_1_temp)) +
geom_boxplot() +
scale_fill_brewer(palette = 'Set1', direction = -1) +
guides(fill = 'none') +
theme_bw(base_size = 15)

#### hub gene: LOC111137329

In [ ]:
geneInfo %>% filter(Gene == 'LOC111137329')

i do not understand how a hub gene isn't in the gene info df ... i've rerun the WGCNA and still am not getting it...

i'll deal with it later

In [ ]:
rownames(black_vst) <- NULL

black.matrix <- black_vst %>%
column_to_rownames('X')

head(black.matrix)

In [ ]:
annotation <- growth %>%
  dplyr::select(Sample, Phase_1_temp) %>%
  column_to_rownames("Sample") 

annotation_colors <- list(
  Phase_1_treat = c(
    Ambient = "burlywood3",
    Warm = "palevioletred"
  )
)

head(annotation)
head(annotation_colors)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 15)

pheatmap(black.matrix,
       # cluster_cols = FALSE,
        annotation_col = annotation,
        annotation_colors = annotation_colors,
         cutree_rows = 2, 
         cutree_cols = 2,
         scale = 'row' # standardizes each gene separately across samples
        ) 

so we see that there are a number of ambient samples that cluster together (mostly control oysters) that have higher expression of this group of genes - when I look back at the module-trait heatmap, I see that there is an almost significant (says 0.05 but that must be rounded) correlation of this module with control conditions - and so I can clearly see here that control oysters are driving this correlation

### blue module
positively correlated with shell growth and temperature treatment

In [ ]:
# pull red genes
blue_genes <- geneInfo %>% 
filter(ModuleColor == 'blue') %>% 
pull(Gene)

length(blue_genes) # 1149 genes in the red module

In [ ]:
# vst for red genes
blue_vst <- vst[vst$X %in% blue_genes,]

dim(blue_vst)
head(blue_vst)

In [ ]:
# convert vst to long format
blue_vst.long <- blue_vst %>%
pivot_longer(cols = -X,
             names_to = 'Sample',
             values_to = 'vst') %>%
dplyr::rename(Gene = X)

head(blue_vst.long)

In [ ]:
# add growth info
blue_vst_growth <- merge(blue_vst.long, growth, by = 'Sample')
head(blue_vst_growth)

#### plot growth vs. module expression

In [ ]:
ggplot(blue_vst_growth, aes(x = Actual_shell_growth_mg, y = vst, col = Phase_1_temp)) +
geom_point() +
geom_smooth() +
theme_bw()

#### hub gene: LOC111118049
[LOC111118049](https://www.ncbi.nlm.nih.gov/gene/?term=LOC111118049): eukaryotic translation initiation factor 2 subunit 1-like 
- enables RNA binding, ribosome binding, translation initiation factory activity
- involved in translational initiation

In [ ]:
options(repr.plot.width = 7.5, repr.plot.height = 5)

blue_hubGene <- blue_vst_growth %>% filter(Gene == 'LOC111118049')

ggplot(blue_hubGene, aes(x = vst, y = Actual_shell_growth_mg, col = Phase_1_treat)) +
geom_point(size = 3) +
scale_color_brewer(palette = 'Set2', direction = -1) +
labs(x = 'VST Expression',
     y = 'Shell Growth (mg)',
     col = 'Treatment',
     title = 'Blue Hub Gene LOC111118049') +
coord_flip() +
theme_bw(base_size = 15)

it does appear that higher expression of this gene does correlate with increased shell growth

#### GS heatmap

but first, need to look at the correlation between treatment and shell growth - because if there is a correlation, then the traits are confounding

In [ ]:
growth$temp_binary <- ifelse(growth$Phase_1_treat == "Warm", 1, 0)
cor.test(growth$Actual_shell_growth_mg, growth$temp_binary)

so appears that there is no significant correlation between shell growth and warm treatment

just want to check out if there is a significant effect of treatment on growth

In [ ]:
summary(aov(Actual_shell_growth_mg ~ Phase_1_temp * Phase_1_DO, data = growth))

In [ ]:
options(repr.plot.height = 7.5, repr.plot.width = 7.5)

growth$Phase_1_treat <- factor(growth$Phase_1_treat, levels = c('Cont', 'Hyp', 'Warm', 'Both'))

shell_growth <- ggplot(growth, aes(x = Phase_1_treat, y = Actual_shell_growth_mg, fill = Phase_1_treat)) +
geom_violin() +
geom_point() +
stat_summary(
    fun = "mean", 
    geom = "point", 
    shape = 8,      # asterick shape
    size = 3,        # Size of the point
    color = "yellow"  # Outline color
  ) +
scale_fill_brewer(palette = 'Set2') +
guides(fill = 'none') +
labs(x = 'Treatment',
     y = 'Shell Growth (mg)',
    title = 'Shell Growth') +
theme_bw(base_size = 15) +
theme(plot.title = element_text(hjust = 0.5))

shell_growth

so it does look like the warming treatment for these oysters do have increased shell growth (more consistently/more of the replicates have higher growth) than the other treatments - but this is not a significant difference

need to pull out gene significance for shell growth and temperature treatment

In [ ]:
blue_gs <- geneInfo %>% 
dplyr::select(Gene, GO.terms, ModuleColor, GS.Actual_shell_growth_mg, GS.warm, p.GS.GS.Actual_shell_growth_mg, p.GS.GS.warm) %>%
filter(ModuleColor == 'blue')

head(blue_gs)

gene significance = correlation (-1:1), strength and direction, between trait and that gene

the corresponding pvalue tells you how likely you'd obverse a correlation that large by chance alone

I want to make a df that pulls genes, and determines if they're temp-associated (warm pval < 0.05), growth-associated (growth pval < 0.05), both, or neither (and therefore a module member only)

In [ ]:
blue_gs2 <- blue_gs %>%
mutate(geneClass = case_when(
    p.GS.GS.Actual_shell_growth_mg < 0.05 & p.GS.GS.warm > 0.05 ~ 'Growth',
    p.GS.GS.Actual_shell_growth_mg > 0.05 & p.GS.GS.warm < 0.05 ~ 'Warm',
    p.GS.GS.Actual_shell_growth_mg > 0.05 & p.GS.GS.warm > 0.05 ~ 'Neither',
    p.GS.GS.Actual_shell_growth_mg < 0.05 & p.GS.GS.warm < 0.05 ~ 'Both'))

head(blue_gs2)

In [ ]:
blue_gs2$geneClass <- factor(blue_gs2$geneClass, levels = c('Both', 'Growth', 'Warm', 'Neither'))

ggplot(blue_gs2, aes(x = geneClass, fill = geneClass)) +
geom_bar() +
geom_text(
    stat = "count", 
    aes(label = after_stat(count)),
    vjust = 0.1
  ) +
scale_fill_brewer(palette = 'Set2') +
guides(fill = 'none') +
labs(x = 'Associated Factor',
     y = 'Number of Genes') +
theme_bw(base_size = 15)

interesting - so the majority of the genes in the blue module are not significantly correlated with the temperature treatment/shell growth, but are instead just members of the module (not sure how that works? it's just that they're connected in the network? and so they're not drivers of the module relationship?)

but we see genes separate into those that are associated with only the warm treatment, only shell growth, and those associated with both shell growth and the warm treatment

should plot heatmap of GS correlations for these and pull out each of the gene classes and look at what they're enriched for 

In [ ]:
blue.matrix <- blue_gs2 %>% dplyr::select(Gene, GS.Actual_shell_growth_mg, GS.warm) %>% column_to_rownames('Gene')
head(blue.matrix)

In [ ]:
options(repr.plot.height = 20, repr.plot.width = 10)

pheatmap(blue.matrix,
         cutree_rows = 2, 
         cutree_cols = 2
        )

would probably be interesting to pull the genes that have opposing patterns for shell growth and warm treatment - those that are dark blue for one factor and dark red for another

looks like there's a similar group of genes that drive the positive correlation between the module and the traits (especially the top portion with red genes)

but then there's a group of genes that have lower expression with increased shell growth, but continue to have higher expression in the warm treatments - what could this mean???? maybe that's why we don't see a correlation between warm and increased shell growth, bc the continued expression of these genes offsets effects of warming? 

Sarah and Teresa want to see a heatmap of expression specifically - so instead of the gene significance correlation, they want to see the correlation of gene expression with shell growth and warm treatment

although thinking this through, I still don't think you can show raw GE for shell at least, because the samples are the same for shell growth and treatment - so you would be asking what the correlation between shell growth and GE is and therefore not plotting raw GE again

#### vst heatmap
plotting heatmap of raw GE of the blue module genes in each sample

In [ ]:
rownames(blue_vst) <- NULL

In [ ]:
# vst matrix
# rows = samples
# genes = columns
blue_vst.matrix <- blue_vst %>%
column_to_rownames('X') 

dim(blue_vst.matrix)
head(blue_vst.matrix)

In [ ]:
annotation <- growth %>%
  dplyr::select(Sample, Phase_1_treat) %>%
  column_to_rownames("Sample") 

annotation_colors <- list(
  Phase_1_treat = c(
    Cont = "burlywood3",
    Warm = "palevioletred",
      Hyp = 'steelblue3',
      Both = 'plum3'
  )
)

head(annotation)
head(annotation_colors)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 15)

pheatmap(blue_vst.matrix,
        cluster_cols = FALSE,
        annotation_col = annotation,
        annotation_colors = annotation_colors,
        # cutree_rows = 5, 
         #cutree_cols = 4,
         scale = 'row' # standardizes each gene separately across samples
        ) 

it looks like the warm treatment oysters have consistently higher expression of this set of genes compared to the other treatments

below: let the heatmap cluster similar samples

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 15)

pheatmap(blue_vst.matrix,
        #cluster_cols = FALSE,
        annotation_col = annotation,
        annotation_colors = annotation_colors,
        # cutree_rows = 5, 
        cutree_cols = 4,
        scale = 'row' # standardizes each gene separately across samples
        ) 

now plotting this for growth - sorting the samples by growth and then showing expression for individuals

In [ ]:
ordered_growth <- growth[order(growth$Actual_shell_growth_mg), ]
head(ordered_growth)

In [ ]:
ordered_blue_vst.matrix <- blue_vst.matrix[,ordered_growth$Sample]
head(ordered_blue_vst.matrix)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 15)

pheatmap(ordered_blue_vst.matrix, 
          annotation_col = annotation,
        annotation_colors = annotation_colors,
         cluster_rows = FALSE, 
         cluster_cols = FALSE,
         scale = 'row' # standardizes each gene separately across samples
)

the left of the heatmap have lower shell growth, the right of the heatmap have higher shell growth

we do see that the majority of the samples with higher shell growth are the warm samples - so again, suggests possible confounding relationship here (but this is interesting because I thought control oysters have the most shell growth out of all of the treatments - altough I plotted it like Sophie where the point is the average and the bars is the standard error)

#### growth associated genes

based on the plots above, pulling out the genes associated with growth and plotting the VST expression against growth

In [ ]:
growth_blue <- blue_gs2 %>%
filter(geneClass == 'Growth')

dim(growth_blue)
head(growth_blue)

In [ ]:
# vst for red genes
growth_blue.vst <- vst[vst$X %in% growth_blue$Gene,]

dim(growth_blue.vst)
head(growth_blue.vst)

In [ ]:
# convert vst to long format
growth_blue.vst.long <- growth_blue.vst %>%
pivot_longer(cols = -X,
             names_to = 'Sample',
             values_to = 'vst') %>%
rename(Gene = X)

head(growth_blue.vst.long)

In [ ]:
# add growth info
growth_blue_meta <- merge(growth_blue.vst.long, growth, by = 'Sample')
head(growth_blue_meta)

In [ ]:
options(repr.plot.height = 5, repr.plot.width = 5)

ggplot(growth_blue_meta, aes(x = Actual_shell_growth_mg, y = vst)) +
geom_point() +
geom_smooth() +
theme_bw()

when considering all of the genes associated with growth in this module, we see a general increase in expression with an increase in shell growth (but it's not super strong)

what if i pull out the top 20 genes with the strongest correlations with growth?

In [ ]:
blue_growth.top20 <- growth_blue[order(growth_blue$GS.Actual_shell_growth_mg, decreasing = TRUE),][1:20,]
head(blue_growth.top20)

In [ ]:
# vst for red genes
growth_blue.vst20 <- vst[vst$X %in% blue_growth.top20$Gene,]

# convert vst to long format
growth_blue.vst20.long <- growth_blue.vst20 %>%
pivot_longer(cols = -X,
             names_to = 'Sample',
             values_to = 'vst') %>%
rename(Gene = X)

# add growth info
growth_blue_meta.top20 <- merge(growth_blue.vst20.long, growth, by = 'Sample')
head(growth_blue_meta.top20)

In [ ]:
ggplot(growth_blue_meta.top20, aes(x = Actual_shell_growth_mg, y = vst)) +
geom_point() +
geom_smooth() +
theme_bw()

In [ ]:
options(repr.plot.height = 10, repr.plot.width = 12.5)

ggplot(growth_blue_meta.top20, aes(x = Actual_shell_growth_mg, y = vst, color = Gene)) +
geom_point() +
geom_smooth() +
guides(color = 'none') +
facet_wrap(~Gene) +
theme_bw(base_size = 15)

so this shows the top 20 genes with the highest gene significance (GS) with pvals only significant (p < 0.05) for shell growth (and not the warm treatment)

there doesn't appear to be a super strong correlation between GE and shell growth, although it is positive and linear. It does appear that the outlier growth point tends to have slightly higher expression in some genes - I should probe what these 20 genes are

In [ ]:
annot[annot$gene_id %in% growth_blue_meta.top20$Gene, ]

cool! so not seeing as many uncharacterized genes as I normally do when I pull out gene names

seeing some genes involved in epigenetic changes (methyltransferase) and spliceosome, also growth factor and biogenesis related genes - which would make sense as being involved in growth

#### warm associated genes

based on the plots above, pulling out the genes associated with growth and plotting the VST expression against growth

In [ ]:
warm_blue <- blue_gs2 %>%
filter(geneClass == 'Warm')

dim(warm_blue)
head(warm_blue)

In [ ]:
# vst for red genes
warm_blue.vst <- vst[vst$X %in% warm_blue$Gene,]

dim(warm_blue.vst)
head(warm_blue.vst)

In [ ]:
colnames(warm_blue.vst)

In [ ]:
# convert vst to long format
warm_blue.vst.long <- warm_blue.vst %>%
pivot_longer(cols = -X,
             names_to = 'Sample',
             values_to = 'vst') 

colnames(warm_blue.vst.long)[1] <- 'Gene'

head(warm_blue.vst.long)

In [ ]:
# add growth info
warm_blue_meta <- merge(warm_blue.vst.long, growth, by = 'Sample')
head(warm_blue_meta)

In [ ]:
ggplot(warm_blue_meta, aes(x = Phase_1_treat, y = vst, fill = Phase_1_treat)) +
geom_boxplot() +
scale_fill_brewer(palette = 'Set2') +
guides(fill = 'none') +
theme_bw(base_size = 15)

when considering all of the genes associated with growth in this module, we see the average expression of these genes are higher in the warm treatment compared to the other treatments

what if i pull out the top 20 genes with the strongest correlations with the warm treatment?

In [ ]:
warm_growth.top20 <- warm_blue[order(warm_blue$GS.warm, decreasing = TRUE),][1:20,]
head(warm_growth.top20)

In [ ]:
warm_blue_meta.top20 <- warm_blue_meta[warm_blue_meta$Gene %in% warm_growth.top20$Gene, ]

length(unique(warm_blue_meta.top20$Gene))
head(warm_blue_meta.top20)

In [ ]:
warm_blue_meta.top20$Phase_1_treat <- factor(warm_blue_meta.top20$Phase_1_treat, levels = c('Cont', 'Hyp', 'Warm', 'Both'))

ggplot(warm_blue_meta.top20, aes(x = Phase_1_treat, y = vst, fill = Phase_1_treat)) +
geom_boxplot() +
scale_fill_brewer(palette = 'Set2') +
guides(fill = 'none') +
theme_bw(base_size = 15)

In [ ]:
options(repr.plot.height = 10, repr.plot.width = 12.5)

ggplot(warm_blue_meta.top20, aes(x = Phase_1_treat, y = vst, fill = Phase_1_treat)) +
geom_boxplot() +
scale_fill_brewer(palette = 'Set2') +
guides(fill = 'none') +
facet_wrap(~Gene) +
theme_bw(base_size = 15)

so this shows the top 20 genes with the highest gene significance (GS) with pvals only significant (p < 0.05) for warm treatment only (and not shell growth)

so some of the genes look like there's small differences in relative expression across treatments, while others are more distinct

In [ ]:
annot[annot$gene_id %in% warm_blue_meta.top20$Gene, ]

looks like a lot of genes related to cell cycle (anaphase promoting complex, zygotic DNA replication licensing factor

are these top warm associated genes driven by only a few replicates, or are they consistenly highly expressed across all warm replicates?

In [ ]:
# pull out only warm associated genes in the blue module VST matrix
blue_warm_VSTmatrix <- blue_vst.matrix[rownames(blue_vst.matrix) %in% warm_blue_meta$Gene,]

dim(blue_warm_VSTmatrix)
head(blue_warm_VSTmatrix)

In [ ]:
options(repr.plot.height = 10, repr.plot.width = 12.5)

pheatmap(blue_warm_VSTmatrix,
        cluster_cols = FALSE,
        annotation_col = annotation,
        annotation_colors = annotation_colors,
         scale = 'row' # standardizes each gene separately across samples
        ) 

In [ ]:
options(repr.plot.height = 10, repr.plot.width = 12.5)

pheatmap(blue_warm_VSTmatrix,
        #cluster_cols = FALSE,
        annotation_col = annotation,
        annotation_colors = annotation_colors,
         cutree_cols = 2,
         scale = 'row' # standardizes each gene separately across samples
        ) 

okay! so the warm associated genes are consistently highly expressed in the warm treatment oysters!!

and what's interesting is that the other replicates/treatments seem to be quite variable in their expression - so these genes are a directed response to experiencing JUST warming

it would be interested to take this gene set and look at the expression/heatmap of expression in phase 2 oysters 

#### growth AND warm associated genes

based on the plots above, pulling out the genes associated with growth and plotting the VST expression against growth/treatment

In [ ]:
both_blue <- blue_gs2 %>%
filter(geneClass == 'Both')

# pull vst expr.
both_blue.vst <- vst[vst$X %in% both_blue$Gene,]

# convert vst to long format
both_blue.vst.long <- both_blue.vst %>%
pivot_longer(cols = -X,
             names_to = 'Sample',
             values_to = 'vst') 

# rename gene column
colnames(both_blue.vst.long)[1] <- 'Gene'

# add growth info
both_blue_meta <- merge(both_blue.vst.long, growth, by = 'Sample')

# check
length(unique(both_blue_meta$Gene))
head(both_blue_meta)

In [ ]:
options(repr.plot.height = 5, repr.plot.width = 12.5)

ggplot(both_blue_meta, aes(x = Actual_shell_growth_mg, y = vst)) +
geom_point() +
facet_wrap(~Phase_1_treat, nrow = 1) +
theme_bw(base_size = 15)

hm okay so there appears to be confounding factor - looks like the warm treatment has higher growth than the other treatments

#### GO for gene subsets

**GROWTH** associated genes in the blue module

In [ ]:
blue_growth.go <- enricher(gene = growth_blue$Gene,
                       universe = geneInfo$Gene,
                       TERM2NAME = term2name,
                       TERM2GENE = term2gene,
                       pvalueCutoff = 0.05)

as.data.frame(blue_growth.go)

growth genes are enriched for rRNA processing, splicing, and ribosomal processes

**WARM TREATMENT** associated genes in the blue module

In [ ]:
blue_warm.go <- enricher(gene = warm_blue$Gene,
                       universe = geneInfo$Gene,
                       TERM2NAME = term2name,
                       TERM2GENE = term2gene,
                       pvalueCutoff = 0.05)

as.data.frame(blue_warm.go)

warm treatment genes are enriched for binding (RNA binding, NAD binding, DNA binding, ATP binding), ATP processes, epigenetic processes (chromatin remodeling), biogenesis (ribosome biogenesis, chain fatty acid biosynthetic process)

**BOTH GROWTH AND WARM TREATMENT** associated genes in the blue module

In [ ]:
blue_both.go <- enricher(gene = both_blue_meta$Gene,
                       universe = geneInfo$Gene,
                       TERM2NAME = term2name,
                       TERM2GENE = term2gene,
                       pvalueCutoff = 0.05)

as.data.frame(blue_both.go)

both growth and warm treatment genes are enriched for binding (RNA binding, mRNA binding, ATP binding) and translational processes

#### summary

There are 1149 genes in the blue module - meaning, there are 1149 genes that are co-expressed. The module eigengene (first principal component; representative gene expression) is significantly correlated with shell growth and the warm treatment (both positive). This means that increases in growth is correlated with higher expression of these genes, and the warm treatment sees higher expression of these genes compared to other treatments.

When I look at the relationship between these oysters and growth, I see that the warm treatment has decreased growth compared to control oysters, but increased compared to both oysters (and roughly the same amount of growth as hypoxic oysters)

When we break this module down further, we see:
1. the raw expression of these genes is consistently higher in the warm treatment compared to the other treatments (which tend to be more inconsistent in their expression of these genes)
2. 111 genes are correlated with **both growth and the warm treatment** (determined by the calculated Gene Significance (GS) value which correlates each gene with the trait, similar to how the WGCNA heatmaps are made but for each individual gene instead of the group. I pulled genes with a significant GS pvalue for both the warm treatment and growth)
    - with ORA, I wanted to know if this subset of genes were enriched for specific functions compared to the entire WGCNA gene set - for blue module genes specifically correlated with *both* warm treatment and growth, we see enrichment for various binding (RNA binding, mRNA binding, ATP binding), translation, and ATP hydrolysis activity.
    - could be that these genes are "compensation genes" (and compenstation only in that they allow some level of growth, despite being stressed by temperature)
3. 154 **growth associated genes** (GS pval is significant for shell growth only, not for warm treatment)
   - these genes are enriched for ribosomal processes and splicing
   - these genes contribute to increased shell growth, but not in a specific treatment (just generic growth genes?)
4. 167 **warm associated genes**
   - these genes are enriched for binding (RNA binding, NAD binding, DNA binding, ATP binding) and biogenesis
5. there are also 717 genes that are not significantly correlated with either factor - so they just belong to the module but don't have a significant relationship with either growth or warm treatment which is interesting ...
    

## 3. Plot growth
adding the shell and tissue plots from above into one plot

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 7.5)

plot_grid(shell_growth, tissue_growth)